In [1]:
import os
from dotenv import load_dotenv
import certifi
from langchain_groq import ChatGroq
from langchain import hub
from langchain.tools import tool
import requests
from langchain.tools.tavily_search import TavilySearchResults
from langchain.agents import AgentExecutor,create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\rjrag\miniconda3\envs\langagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["SSL_CERT_FILE"]=certifi.where()
load_dotenv()

GEMINI_API_KEY=os.environ["GEMINI_API_KEY"]
TAVILY_API_KEY=os.environ["TAVILY_API_KEY"]
WEATHERSTACK_API_KEY=os.environ["WEATHERSTACK_API"]

In [3]:
# tool1
search_tool=TavilySearchResults(max_results=2)

# tool2
@tool
def weather_tool(city:str)->str:
    """Fetch current weather information"""
    url = (
    f"https://api.weatherstack.com/current?"
    f"access_key={WEATHERSTACK_API_KEY}&query={city}")

    response=requests.get(url=url)
    data=response.json()

    if "current" not in data:
        return "Data not found"

    return (f"city: {city}"
            f"Weather: {data['current']['temperature']} celsius"
            f"Humidity: {data['current']['humidity']} %")

# LLM
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.5,google_api_key=GEMINI_API_KEY)

# prompt
prompt=hub.pull("hwchase17/react")

# tools collection
tools=[search_tool,weather_tool]

#create agent 
agent=create_react_agent(llm=llm,prompt=prompt,tools=tools)

# agent_execution
agent_exc=AgentExecutor(agent=agent,tools=tools,verbose=True)

c:\Users\rjrag\miniconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [4]:
response=agent_exc.invoke({"input":("find capital of pakistan and find its weather")})
print(response)



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: capital of Pakistan[{'url': 'https://en.wikipedia.org/wiki/Islamabad', 'content': 'Islamabad is the capital city of Pakistan. It is the country\'s tenth-most populous city with a population of over 1.1 million and is federally administered by the Pakistani government as part of the Islamabad Capital Territory — with a metropolitan population of over 2.3 million. Built as a planned city along the Margalla Hills in the 1960s and established in 1967, Islamabad replaced Karachi as Pakistan\'s national capital. It is located in the northern Punjab region, north of the city of Rawalpindi — with which it forms a metropolitan area of over 5.7 million inhabitants. [...] Edit links\n\n Article\n Talk\n\n Read\n View source\n View history\n\nTools\n\nActions\n\n Read\n View source\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page